# Project 8: Clinical Trial Analyzer (CARDIO-BETA Phase II/III Trial)
**School of Engineering, JG University | BTech CSE — Probability & Statistics**

This notebook reproduces the full pipeline with fixed random seed `SEED = 42`: data generation, SQLite ETL & analytical queries, assumption checks, hypothesis testing, Theory vs. Experiment Monte Carlo power validation, and informative dropout bias analysis.


In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, '../src')
from generate_data import generate_clinical_trial_data, SEED
from db_pipeline import build_database_and_clean
from stats_analysis import run_statistical_analysis
from power_simulation import run_power_validation
from dropout_analysis import run_dropout_bias_analysis
print(f'Pipeline modules loaded successfully (SEED = {SEED})')


Pipeline modules loaded successfully (SEED = 42)


## 1. Data Generation, SQLite ETL & Data-Quality Audit


In [2]:
project_dir = Path('..').resolve()
gen_info = generate_clinical_trial_data(project_dir / 'data', seed=SEED)
db_info = build_database_and_clean(project_dir)
print('Patients:', gen_info['n_patients'], '| Visits:', gen_info['n_visits'], '| Raw measurements:', gen_info['n_raw_measurements'])
for ev in db_info['qc_events']:
    print(f" - {ev['issue']}: {ev['count']} ({ev['action_taken']})")


Patients: 300 | Visits: 900 | Raw measurements: 851
 - Duplicate measurement rows: 4 (Retained first record; removed exact duplicates before DB load.)
 - Physiologically impossible SBP values: 2 (Flagged is_valid=0 (ERR_SBP_RANGE) and excluded from primary SBP analysis.)
 - Negative hs-CRP lab measurement: 1 (Flagged is_valid=0 (ERR_CRP_NEGATIVE) and excluded from biomarker analysis.)
 - Missing Week-6 hs-CRP lab draws: 5 (Retained row with NULL crp_mg_l; kept valid SBP/DBP readings.)
 - High hs-CRP clinical outliers (>18 mg/L): 8 (Retained (is_valid=1) and analyzed using non-parametric Mann-Whitney U / Kruskal-Wallis.)
 - Missing Week-12 follow-up visits (dropouts): 53 (Evaluated via Complete-Case, LOCF, and BOCF sensitivity analyses (Stretch Goal).)


## 2. Core Statistical Analysis (Primary SBP, Skewed hs-CRP, 3-Arm ANOVA)


In [3]:
stat_res = run_statistical_analysis(project_dir)
w = stat_res['primary_two_arm']['welch_t']
d = stat_res['primary_two_arm']['cohens_d']
print(f"Primary Two-Arm Welch t-test (High-Dose vs Placebo ΔSBP): diff = {w['mean_diff']:.2f} mmHg, 95% CI [{w['ci_low']:.2f}, {w['ci_high']:.2f}], t = {w['t_stat']:.2f}, p = {w['p_value']:.2e}")
print(f"Effect Size Cohen's d = {d['d']:.3f} (95% CI [{d['ci_low']:.3f}, {d['ci_high']:.3f}])")
print(f"Skewed hs-CRP Mann-Whitney U: U = {stat_res['secondary_crp']['mann_whitney_high_vs_placebo']['U_stat']:.1f}, p = {stat_res['secondary_crp']['mann_whitney_high_vs_placebo']['p_value']:.4f}")
print(f"3-Arm One-Way ANOVA (ΔSBP): F = {stat_res['anova_3arm']['F_stat']:.2f}, p = {stat_res['anova_3arm']['p_value']:.2e}, eta^2 = {stat_res['anova_3arm']['eta_squared']:.3f}")


Primary Two-Arm Welch t-test (High-Dose vs Placebo ΔSBP): diff = -9.55 mmHg, 95% CI [-12.65, -6.45], t = -6.09, p = 1.09e-08
Effect Size Cohen's d = -0.971 (95% CI [-1.293, -0.649])
Skewed hs-CRP Mann-Whitney U: U = 851.0, p = 0.0000
3-Arm One-Way ANOVA (ΔSBP): F = 19.26, p = 1.71e-08, eta^2 = 0.136


## 3. Theory vs. Experiment — Monte Carlo Power Validation


In [4]:
pwr_res = run_power_validation(project_dir, n_sims=2000, seed=SEED)
print('Empirical Type I Error (null d=0, alpha=0.05):', pwr_res['empirical_type1_error_null'])
print('Mean |Empirical - Theoretical| Power Gap:', pwr_res['mean_abs_gap_normal'])
print('Required n/arm for 80% power:', pwr_res['required_n_per_arm_80pct'])


Empirical Type I Error (null d=0, alpha=0.05): 0.0514
Mean |Empirical - Theoretical| Power Gap: 0.0061
Required n/arm for 80% power: {'d_0.3': 176, 'd_0.5': 64, 'd_0.8': 26}


## 4. Stretch Goal — Informative Dropout & Attrition Bias


In [5]:
drp_res = run_dropout_bias_analysis(project_dir)
for row in drp_res['method_comparison']:
    print(f"{row['method']:30s} | Diff: {row['mean_diff_mmhg']:6.2f} mmHg [{row['ci_low']:6.2f}, {row['ci_high']:6.2f}] | Bias: {row['bias_vs_truth_mmhg']:+5.2f} mmHg")


True Full Cohort (Benchmark)   | Diff:  -8.65 mmHg [-11.64,  -5.67] | Bias: +0.00 mmHg
Complete-Case (CCA)            | Diff:  -9.55 mmHg [-12.65,  -6.45] | Bias: -0.90 mmHg
LOCF (Week-6 Carried)          | Diff:  -7.70 mmHg [-10.50,  -4.90] | Bias: +0.95 mmHg
BOCF (Zero Change Imputed)     | Diff:  -6.51 mmHg [ -9.32,  -3.70] | Bias: +2.14 mmHg
